# 05_preparacao_pronaf_rotulo_pu

## 1. Objetivo, granularidade e saída

Este notebook prepara a base do PRONAF para criação do rótulo observado de PU Learning.

- **Objetivo**: gerar rótulo observado de acesso ao PRONAF nos últimos 5 anos.
- **Granularidade**: nível **pessoa** (chave `cpf_agricultor`).
- **Janela temporal**: operações de **2021 a 2025** (referência: `2025-12-31`).
- **Saída**: `dados/dados_processados/pronaf_rotulo_pu_5anos.parquet`.

No contexto de PU Learning, `s = 1` indica **positivo observado** (CPF com operação no recorte).


## 2. Imports e configuração


In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

INPUT_PATH = ROOT / "dados" / "dados_brutos" / "pronaf.parquet"
OUTPUT_PATH = ROOT / "dados" / "dados_processados" / "pronaf_rotulo_pu_5anos.parquet"

DATA_REFERENCIA = pd.Timestamp("2025-12-31")
ANO_INICIAL = 2021
ANO_FINAL = 2025

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"Arquivo não encontrado: {INPUT_PATH}")

schema_cols = set(pq.ParquetFile(INPUT_PATH).schema.names)
colunas_valor_candidatas = ["VL_OPERACAO", "VL_PARC_CREDITO"]
COLUNA_VALOR_OPERACAO = next((c for c in colunas_valor_candidatas if c in schema_cols), None)
if COLUNA_VALOR_OPERACAO is None:
    raise ValueError(
        "Nenhuma coluna de valor encontrada entre as candidatas: "
        f"{colunas_valor_candidatas}"
    )

COLUNAS_PRONAF = ["CD_CPF_CNPJ", "ANO", COLUNA_VALOR_OPERACAO, "CD_PROGRAMA"]

print(f"ROOT: {ROOT}")
print(f"INPUT_PATH: {INPUT_PATH}")
print(f"OUTPUT_PATH: {OUTPUT_PATH}")
print(f"Janela temporal: {ANO_INICIAL}-{ANO_FINAL} | referência: {DATA_REFERENCIA.date()}")
print(f"Coluna de valor utilizada: {COLUNA_VALOR_OPERACAO}")


## 3. Leitura do PRONAF com colunas mínimas


In [ ]:
df_pronaf = pd.read_parquet(
    INPUT_PATH,
    columns=COLUNAS_PRONAF,
    engine="pyarrow",
)

if COLUNA_VALOR_OPERACAO != "VL_OPERACAO":
    df_pronaf = df_pronaf.rename(columns={COLUNA_VALOR_OPERACAO: "VL_OPERACAO"})

print(f"Shape inicial: {df_pronaf.shape}")
display(df_pronaf.head(3))
df_pronaf.info(show_counts=True)


## 4. Filtro de programa e normalização de CPF


In [ ]:
linhas_iniciais = len(df_pronaf)

df_pronaf["CD_PROGRAMA"] = (
    df_pronaf["CD_PROGRAMA"]
    .astype("string")
    .str.strip()
    .str.zfill(4)
)

df_pronaf = df_pronaf[df_pronaf["CD_PROGRAMA"] == "0001"].copy()
linhas_pos_programa = len(df_pronaf)

df_pronaf["CD_CPF_CNPJ"] = (
    df_pronaf["CD_CPF_CNPJ"]
    .astype("string")
    .str.replace(r"\D", "", regex=True)
    .str.strip()
    .str.zfill(11)
)

is_cpf = df_pronaf["CD_CPF_CNPJ"].str.len().eq(11).fillna(False)
df_pronaf = df_pronaf[is_cpf].copy()
linhas_pos_cpf = len(df_pronaf)

df_pronaf = df_pronaf.rename(columns={"CD_CPF_CNPJ": "cpf_agricultor"})
df_pronaf["cpf_agricultor"] = df_pronaf["cpf_agricultor"].astype("string")

print(f"Linhas iniciais: {linhas_iniciais:,}")
print(f"Após filtro CD_PROGRAMA == '0001': {linhas_pos_programa:,}")
print(f"Após filtro de CPF (11 dígitos): {linhas_pos_cpf:,}")
print(f"CPFs únicos (parcial): {df_pronaf['cpf_agricultor'].nunique():,}")


## 5. Janela temporal e tratamento de `VL_OPERACAO`


In [ ]:
df_pronaf["ANO"] = pd.to_numeric(df_pronaf["ANO"], errors="coerce")
anos_invalidos = int(df_pronaf["ANO"].isna().sum())

mask_janela = df_pronaf["ANO"].between(ANO_INICIAL, ANO_FINAL, inclusive="both")
df_pronaf = df_pronaf[mask_janela.fillna(False)].copy()

df_pronaf["VL_OPERACAO"] = pd.to_numeric(df_pronaf["VL_OPERACAO"], errors="coerce")
vl_operacao_invalidos = int(df_pronaf["VL_OPERACAO"].isna().sum())
df_pronaf["VL_OPERACAO"] = df_pronaf["VL_OPERACAO"].fillna(0.0)

print(f"ANOs inválidos (coerce): {anos_invalidos:,}")
print(f"Shape após filtro temporal {ANO_INICIAL}-{ANO_FINAL}: {df_pronaf.shape}")
print(f"VL_OPERACAO inválido/ausente tratado com 0.0: {vl_operacao_invalidos:,}")
display(df_pronaf[["ANO", "VL_OPERACAO"]].describe().T)


In [ ]:
df_pronaf

## 6. Agregação por CPF e criação de `s`


In [ ]:
df_pronaf_rotulo = (
    df_pronaf
    .groupby("cpf_agricultor", as_index=False)
    .agg(
        n_ops_pronaf_5a=("cpf_agricultor", "size"),
        valor_total_pronaf_5a=("VL_OPERACAO", "sum"),
        ano_ult_op_pronaf=("ANO", "max"),
    )
)

df_pronaf_rotulo["s"] = 1

print(f"Shape agregado por CPF: {df_pronaf_rotulo.shape}")
display(df_pronaf_rotulo.head(5))


## 7. Tipagem final e validação mínima de qualidade


In [ ]:
df_pronaf_rotulo["cpf_agricultor"] = df_pronaf_rotulo["cpf_agricultor"].astype("string")
df_pronaf_rotulo["n_ops_pronaf_5a"] = df_pronaf_rotulo["n_ops_pronaf_5a"].astype("int32")
df_pronaf_rotulo["valor_total_pronaf_5a"] = pd.to_numeric(
    df_pronaf_rotulo["valor_total_pronaf_5a"], errors="coerce"
).astype("float64")

df_pronaf_rotulo["ano_ult_op_pronaf"] = pd.to_numeric(
    df_pronaf_rotulo["ano_ult_op_pronaf"], errors="coerce"
).astype("int16")

df_pronaf_rotulo["s"] = df_pronaf_rotulo["s"].astype("int8")

print(f"Shape final: {df_pronaf_rotulo.shape}")

print("Missing em colunas-chave:")
display(
    df_pronaf_rotulo[[
        "cpf_agricultor",
        "n_ops_pronaf_5a",
        "valor_total_pronaf_5a",
        "ano_ult_op_pronaf",
        "s",
    ]].isna().sum()
)

duplicados_cpf = int(df_pronaf_rotulo.duplicated(subset=["cpf_agricultor"]).sum())
print(f"Duplicatas de cpf_agricultor: {duplicados_cpf:,}")

anos_fora_janela = int((~df_pronaf_rotulo["ano_ult_op_pronaf"].between(ANO_INICIAL, ANO_FINAL)).sum())
print(f"Registros com ano_ult_op_pronaf fora da janela {ANO_INICIAL}-{ANO_FINAL}: {anos_fora_janela:,}")

print("Distribuição de s:")
display(df_pronaf_rotulo["s"].value_counts(dropna=False).sort_index())

df_pronaf_rotulo.info(show_counts=True)


## 8. Persistência


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
df_pronaf_rotulo.to_parquet(OUTPUT_PATH, index=False)

print(f"Arquivo salvo em: {OUTPUT_PATH}")
print(f"Shape salvo: {df_pronaf_rotulo.shape}")
print(f"Total de colunas: {len(df_pronaf_rotulo.columns)}")
display(df_pronaf_rotulo.columns.to_list())
